# Week 2 — Preprocess images + baseline Tesseract gap analysis

This notebook replaces the handout's grayscale -> squash-resize -> hard-threshold
pipeline with `src/preprocessing.py`'s deskew + adaptive-contrast + letterbox
pipeline (see that file's docstring for *why* — short version: a fixed-size
squash and a global binary threshold both destroy information a transformer
OCR model needs). It then runs the same Tesseract baseline the handout asks
for, but scores it with CER/WER instead of eyeballing 5 outputs.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- pulling latest instead of re-cloning.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

!pip install -q --upgrade -r requirements.txt
# Force transformers (and only transformers -- not the whole requirements
# file, so this doesn't also re-download the much larger torch wheel) to
# the pinned exact version. A plain `-r requirements.txt` install can
# leave an already-installed version in place if pip's resolver treats it
# as already 'satisfying' things.
!pip install -q --upgrade --force-reinstall "transformers==4.46.3"

# Ground-truth check via subprocess (`pip show`), NOT `import transformers`
# -- if a cell anywhere earlier in this session already imported
# transformers, Python caches that in memory for the rest of the process,
# and reinstalling on disk doesn't change what's already loaded here. The
# actual training/eval scripts run as a separate `!python ...` subprocess
# each time, so they always see what's genuinely on disk -- this mirrors
# that instead of this notebook kernel's own (possibly stale) view.
result = subprocess.run(['pip', 'show', 'transformers'], capture_output=True, text=True)
installed_version = next(
    (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
    '?',
)
print('transformers installed on disk:', installed_version)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

if not installed_version.startswith('4.'):
    raise RuntimeError(
        f'transformers on disk is {installed_version}, not the pinned 4.46.3 -- '
        'pip could not satisfy the pin here, likely another package in this '
        'Colab image requires transformers>=5. Re-run this cell and share the '
        'full pip output above so the real conflicting package can be found.'
    )
print('Setup complete -- transformers', installed_version, 'is correctly installed on disk.')


## Part A — Preprocess every collected image

In [ ]:
from preprocessing import preprocess_folder, PreprocessConfig

count = preprocess_folder('data/raw', 'data/processed', PreprocessConfig())
print(f'Preprocessed {count} images -> data/processed/')


In [ ]:
# Visual sanity check: compare a few raw vs. processed images.
import glob
from PIL import Image
import matplotlib.pyplot as plt

raw_samples = sorted(glob.glob('data/raw/**/*.png', recursive=True))[:3] + \
              sorted(glob.glob('data/raw/**/*.jpg', recursive=True))[:3]
fig, axes = plt.subplots(len(raw_samples[:3]), 2, figsize=(10, 3 * len(raw_samples[:3])))
for i, raw_path in enumerate(raw_samples[:3]):
    import os
    processed_path = os.path.join('data/processed', os.path.basename(raw_path))
    axes[i, 0].imshow(Image.open(raw_path)); axes[i, 0].set_title('raw'); axes[i, 0].axis('off')
    if os.path.exists(processed_path):
        axes[i, 1].imshow(Image.open(processed_path)); axes[i, 1].set_title('processed'); axes[i, 1].axis('off')
plt.tight_layout()
plt.show()


## Part B — Baseline Tesseract OCR + quantified gap analysis

Install Tesseract's Urdu language pack first (Colab shell command):

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-urd
!pip install -q pytesseract

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'scripts/tesseract_baseline.py',
     '--labels-csv', 'data/labels.csv',
     '--report-csv', 'reports/tesseract_baseline.csv'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## Why We Need a Better Model

*(copy this section, filled in with your printed CER%/examples above, into your GitHub README)*

Tesseract fails on Urdu because... its Urdu language model was trained on a much
smaller and less varied dataset than its English one, and it does not handle
Nastaliq's diagonal letter-stacking and heavy contextual joining well — it
tends to segment the image into a flat left-to-right strip of isolated glyph
guesses, which breaks down badly whenever letters overlap vertically or a
word's letterforms change with position. The character error rate printed
above (see `reports/tesseract_baseline.csv` for every example) quantifies
just how large that gap is, which is exactly the gap Weeks 3-4's fine-tuned
model is built to close.
